# Whiteboard

Notes and scratch work for this chapter.

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.21
torch version: 2.11.0
tokenizers version: 0.22.2


## Exercise 5.1:

In [ ]:
from pathlib import Path
import time

from reasoning_from_scratch.ch05 import heuristic_score


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=2048,
    verbose=False,
    prompt_suffix="",
    temperature=1.0,
    top_p=1.0,
    seed=None,
    num_samples=10,
):
    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"]) + prompt_suffix

            results = self_consistency_vote(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device,
                num_samples=num_samples,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                show_progress=False,
                show_long_answer=False,
                seed=seed,
            )

            # Majority vote winner available
            if results["final_answer"] is not None:
                extracted = results["final_answer"]

            
            else:
                best = None
                best_score = float("-inf")
            
                for cand in results["majority_winners"]:
                    scores = [
                        heuristic_score(results["full_answers"][idx], prompt=prompt)
                        for idx in results["groups"][cand]
                    ]
            
                    score = max(scores)
            
                    if score > best_score:
                        best_score = score
                        best = cand
            
                extracted = best

            

    
    return num_correct, num_examples, acc


## Exercise 5.2:

In [ ]:
from reasoning_from_scratch.ch05 import (
    heuristic_score
)

def self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device,
    num_samples=10,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    show_progress=True,
    show_long_answer=False,
    seed=None,
):
    full_answers, short_answers = [], []
    counts = Counter()
    groups = {}
    majority_winners, final_answer = [], None
    best_score, best_idx = float("-inf"), None

    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        short = extract_final_candidate(answer, fallback="number_then_full")
        full_answers.append(answer)
        short_answers.append(short)
        counts[short] += 1

        if short in groups:
            groups[short].append(i)
        else:
            groups[short] = [i]

        score = heuristic_score(answer, prompt=prompt)

        if score > best_score:
            best_score, best_idx = score, i

        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    if best_idx is not None:
        final_answer = short_answers[best_idx]
        majority_winners = [final_answer]

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }



## Exercise 5.3:

In [4]:
from reasoning_from_scratch.ch05 import avg_logprob_answer


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=2048,
    verbose=False,
    prompt_suffix="",
    temperature=1.0,
    top_p=1.0,
    seed=None,
    num_samples=10,
):
    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"]) + prompt_suffix

            results = self_consistency_vote(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device,
                num_samples=num_samples,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                show_progress=False,
                show_long_answer=False,
                seed=seed,
            )

            # Majority vote winner available
            if results["final_answer"] is not None:
                extracted = results["final_answer"]

            
            else:
                best = None
                best_score = float("-inf")
            
                # Consider all members of each majority group
                for cand in results["majority_winners"]:
                    scores = []
            
                    for idx in results["groups"][cand]:
                        candidate_full = results["full_answers"][idx]
            
                        score = avg_logprob_answer(
                            model=model,
                            tokenizer=tokenizer,
                            prompt=prompt,
                            answer=candidate_full,
                            device=device,
                        )
                        scores.append(score)
            
                    cand_score = max(scores)
            
                    if cand_score > best_score:
                        best_score = cand_score
                        best = cand
            
                extracted = best
            

    
    return num_correct, num_examples, acc


## Exercise 5.4:

In [5]:
from reasoning_from_scratch.ch05 import (
    avg_logprob_answer
)


def self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device,
    num_samples=10,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    show_progress=True,
    show_long_answer=False,
    seed=None,
):
    full_answers, short_answers = [], []
    counts = Counter()
    groups = {}
    majority_winners, final_answer = [], None
    best_score, best_idx = float("-inf"), None

    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        short = extract_final_candidate(answer, fallback="number_then_full")
        full_answers.append(answer)
        short_answers.append(short)
        counts[short] += 1

        if short in groups:
            groups[short].append(i)
        else:
            groups[short] = [i]

            score = avg_logprob_answer(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                answer=answer,
                device=device
            )
        if score > best_score:
            best_score, best_idx = score, i

        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    if best_idx is not None:
        final_answer = short_answers[best_idx]
        majority_winners = [final_answer]

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }


## Exercise 5.5:

In [ ]:
torch.manual_seed(0)

results_logprob = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=heuristic_score,  # NEW
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)
